# 学生管理系统（数据库版）项目笔记

> 基于 Python + SQLAlchemy + PyMySQL 的三层架构 CLI 学生管理系统
> 完整记录：架构设计、搭建流程、踩坑排查、最佳实践、扩展方向

---

## 目录

- [一、项目整体逻辑](#一项目整体逻辑)
- [二、项目目录结构](#二项目目录结构)
- [三、项目搭建顺序](#三项目搭建顺序)
- [四、各层职责与模板](#四各层职责与模板)
- [五、踩过的坑 & 解决方案](#五踩过的坑--解决方案)
- [六、关键设计原则](#六关键设计原则)
- [七、快速自查清单](#七快速自查清单)
- [八、常用代码片段](#八常用代码片段)
- [九、项目可扩展方向](#九项目可扩展方向)
- [十、一句话总结](#十一句话总结)

---

## 一、项目整体逻辑

### 1.1 分层架构

```
┌─────────────────────────────────────────────┐
│                  main.py                    │  CLI 用户交互
│         登录 / 菜单 / 输入输出 / 异常展示         │
└────────────────────┬────────────────────────┘
                     │ 调用
┌────────────────────▼────────────────────────┐
│                 service/                    │  业务逻辑层
│   参数校验 / 唯一性校验 / 关联校验 / 填时间        │
│   抛 core.exceptions 里的业务异常              │
└────────────────────┬────────────────────────┘
                     │ 调用
┌────────────────────▼────────────────────────┐
│                   dao/                      │  数据访问层
│   拼 SQL / execute / commit / rollback       │
│   把 ORM Row 转成 model 对象                   │
└────────────────────┬────────────────────────┘
                     │ 依赖
┌────────────────────▼────────────────────────┐
│                  model/                     │  数据模型层
│   @dataclass 纯数据类 + from_dict             │
└─────────────────────────────────────────────┘

        横切关注点：
        ├── utils/db_util.py    连接池 + Session 生命周期
        ├── utils/env_util.py   环境变量加载
        ├── core/logger.py      日志
        └── core/exceptions.py  异常体系
```

### 1.2 一次完整调用链（以"新增学生"为例）

```
main.py  student_add()
  │ 收集输入 → Student 对象
  ▼
StudentService.add(student)
  │ 1. 必填校验（姓名、学号）
  │ 2. 学号唯一性校验 → StudentDao.select_by_stu_num
  │ 3. 班级存在性校验 → ClazzDao.select_by_id
  │ 4. 年龄 / 手机号格式校验
  │ 5. 填 create_time / update_time
  ▼
StudentDao.insert(student)
  │ session = DBUtil.get_session()      ← 从连接池拿
  │ session.execute(text(sql), params)  ← 参数绑定
  │ session.commit()                    ← 落库
  │ student.id = result.lastrowid       ← 回填主键
  │ finally: session.close()            ← 归还连接
  ▼
回到 main，打印 student.id
```

### 1.3 职责边界

| 层 | 该做 | 不该做 |
|---|---|---|
| model | 定义字段、`from_dict` | 任何数据库/SQL |
| dao | 拼 SQL、execute、commit/rollback/close、异常包装 | 业务校验、填时间 |
| service | 参数校验、唯一性/关联校验、填时间、调多个 dao | 直接拼 SQL、管理 session |
| main | 输入输出、异常展示 | 业务逻辑 |
| utils/core | 横切工具 | 具体业务 |

---

## 二、项目目录结构

```
05_学生管理系统_数据库/
├── .env                          # 环境变量
├── main.py                       # CLI 入口
├── README.md
│
├── core/                         # 核心工具
│   ├── __init__.py
│   ├── exceptions.py             # 异常体系
│   └── logger.py                 # 日志
│
├── utils/                        # 通用工具
│   ├── __init__.py
│   ├── db_util.py                # 数据库工具（连接池 + session）
│   └── env_util.py               # 环境变量工具
│
├── model/                        # 数据模型
│   ├── __init__.py
│   ├── clazz.py
│   ├── student.py
│   ├── course.py
│   ├── score.py
│   └── user.py
│
├── dao/                          # 数据访问
│   ├── __init__.py
│   ├── clazz_dao.py
│   ├── student_dao.py
│   ├── course_dao.py
│   ├── score_dao.py
│   └── user_dao.py
│
├── service/                      # 业务逻辑
│   ├── __init__.py
│   ├── clazz_service.py
│   ├── student_service.py
│   ├── course_service.py
│   ├── score_service.py
│   └── user_service.py
│
└── logs/                         # 日志目录
    └── student.log
```

---

## 三、项目搭建顺序

```
1. 设计数据库表 → 建库 SQL
2. 写 .env             → 数据库 + 日志配置
3. 写 utils/env_util   → 读环境变量
4. 写 core/exceptions  → 定义异常体系
5. 写 core/logger      → 日志（依赖 env_util）
6. 写 utils/db_util    → 连接池 + get_session
7. 每张表：
   7.1 写 model/X.py
   7.2 写 dao/X_dao.py
   7.3 写 service/X_service.py（先 add + get_by_id，跑通）
   7.4 补全其他方法
8. 最后写 main.py 串起来
```

**核心原则：从下往上写，依赖先行。** 上层调下层，下层不依赖上层。

---

## 四、各层职责与模板

### 4.1 `model` 层

纯数据类，不依赖任何业务/数据库。用 `@dataclass` 声明字段，`from_dict` 从数据库 Row 构造。

```python
from dataclasses import dataclass
from datetime import datetime


@dataclass
class Student:
    """学生类"""
    id: int | None = None
    stu_num: str | None = None
    name: str | None = None
    gender: str | None = None
    age: int | None = None
    class_id: int | None = None
    phone: str | None = None
    create_time: datetime | None = None
    update_time: datetime | None = None

    @staticmethod
    def from_dict(dic):
        """从字典到对象"""
        if dic is None:
            return None
        return Student(**dic)
```

**要点**：
- 所有字段默认 `None`
- 类型注解用 `X | None`
- `from_dict` 统一处理 `None` 输入

### 4.2 `dao` 层模板

```python
from model.student import Student
from utils.db_util import DBUtil
from sqlalchemy import text

from core.exceptions import DatabaseError
from core import logger

log = logger.get_logger()


class StudentDao:
    """学生数据库操作类"""

    _COLUMNS = ("`id`, `stu_num`, `name`, `gender`, `age`, "
                "`class_id`, `phone`, `create_time`, `update_time`")

    @staticmethod
    def insert(student: Student) -> int:
        """写入学生记录"""
        session = None
        try:
            sql = ("INSERT INTO `t_student`"
                   "(`stu_num`, `name`, `gender`, `age`, `class_id`, `phone`, "
                   " `create_time`, `update_time`) "
                   "VALUES (:stu_num, :name, :gender, :age, :class_id, :phone, "
                   "        :create_time, :update_time)")
            params = {
                "stu_num": student.stu_num,
                "name": student.name,
                "gender": student.gender,
                "age": student.age,
                "class_id": student.class_id,
                "phone": student.phone,
                "create_time": student.create_time,
                "update_time": student.update_time,
            }
            session = DBUtil.get_session()
            result = session.execute(text(sql), params)
            session.commit()                          # 关键：提交事务
            student.id = result.lastrowid             # 回填自增主键
            return result.rowcount                    # 返回受影响行数
        except Exception as e:
            if session:
                session.rollback()                    # 失败回滚
            log.exception("插入学生记录失败")           # 自动带堆栈
            raise DatabaseError(e) from e             # 保留原因链
        finally:
            if session:
                session.close()                       # 归还连接池

    @staticmethod
    def select_by_id(id: int) -> Student:
        """按主键查询"""
        session = None
        try:
            sql = f"SELECT {StudentDao._COLUMNS} FROM `t_student` WHERE `id` = :id"
            params = {"id": id}
            session = DBUtil.get_session()
            result = session.execute(text(sql), params)
            row = result.mappings().fetchone()
            return Student.from_dict(row)
        except Exception as e:
            log.exception("查询学生记录失败")
            raise DatabaseError(e) from e
        finally:
            if session:
                session.close()
```

**要点**：
- 每个 DML 方法 `try / except / finally` 三件套
- insert 后必须 `commit` + 回填主键 + 返回 rowcount
- 查询用 `_COLUMNS` 常量
- 异常包装 `raise DatabaseError(e) from e`

### 4.3 `service` 层模板

```python
from datetime import datetime

from model.student import Student
from dao.student_dao import StudentDao
from dao.clazz_dao import ClazzDao
from core.exceptions import ValidationError, AlreadyExistsError, NotFoundError, BusinessException
from core import logger

log = logger.get_logger()


class StudentService:
    """学生业务逻辑层"""

    @staticmethod
    def add(student: Student) -> int:
        """新增学生"""
        # 1. 必填校验
        if student is None:
            raise ValidationError("学生信息不能为空")
        if not student.name or not student.name.strip():
            raise ValidationError("学生姓名不能为空", field="name")
        if not student.stu_num or not student.stu_num.strip():
            raise ValidationError("学号不能为空", field="stu_num")

        # 2. 唯一性校验
        if StudentDao.select_by_stu_num(student.stu_num) is not None:
            raise AlreadyExistsError("学生", field="stu_num", value=student.stu_num)

        # 3. 关联存在性校验
        if student.class_id is not None:
            if ClazzDao.select_by_id(student.class_id) is None:
                raise NotFoundError("班级", student.class_id)

        # 4. 时间字段
        now = datetime.now()
        student.create_time = now
        student.update_time = now

        # 5. 调用 DAO
        row = StudentDao.insert(student)
        if row <= 0:
            raise BusinessException("新增学生记录失败")
        log.info("新增学生成功 id=%s stu_num=%s", student.id, student.stu_num)
        return row
```

**要点**：
- 顺序：必填 → 唯一性 → 关联 → 时间 → 调 DAO
- add/update 共用校验逻辑可抽 `_validate_xxx`
- 用 `core.exceptions` 里的异常，不用 `ValueError`

### 4.4 `main` 层模板

```python
def student_add():
    """新增学生"""
    student = Student()
    student.stu_num = input_str("学号: ")
    student.name = input_str("姓名: ")
    student.gender = input_str("性别: ")
    student.age = input_int("年龄(可空): ", allow_empty=True)
    student.class_id = input_int("班级ID(可空): ", allow_empty=True)
    student.phone = input_str("手机号: ")
    StudentService.add(student)
    print(f"新增学生成功，ID={student.id}")


def student_menu():
    """学生管理子菜单"""
    while True:
        print("1. 新增  2. 查询  3. 更新  4. 删除  0. 返回")
        choice = input_str("请选择: ")
        try:
            if choice == "1":
                student_add()
            # ...
        except AppException as e:
            print(f"[业务错误] {e.message}")
        except ValueError as e:
            print(f"[输入错误] {e}")
        except Exception as e:
            print(f"[系统错误] {e}")
        pause()
```

---

## 五、踩过的坑 & 解决方案

### A. 环境配置类

#### A1. `int() argument must be ... not 'NoneType'`

- **现象**：启动时 `DB_PORT = int(os.getenv("DB_PORT"))` 报错
- **原因**：`.env` 里没这一项，或 `load_dotenv()` 没找到 `.env`
- **解决**：
  1. 确认 `.env` 在运行目录（`os.getcwd()`）
  2. 给 `getenv` 加默认值 `os.getenv("DB_PORT", "3306")`
- **避免**：
  - `.env` 放在项目根目录，和 `main.py` 同级
  - 给所有变量加默认值或启动时校验

#### A2. key 名字对不上

- **现象**：`.env` 写 `DB_NAME`，代码读 `DB_DATABASE` → 读到 `None`
- **原因**：命名约定不统一
- **解决**：统一改成 `DB_DATABASE`
- **避免**：
  - 项目开始时定好 key 命名规范，写在 README
  - `env_util.py` 加校验，读不到直接抛错

```python
def _require(key):
    val = os.getenv(key)
    if not val:
        raise RuntimeError(f"环境变量 {key} 未设置")
    return val
```

#### A3. `Unknown database 'none'`

- **现象**：SQLAlchemy 连库报 `1049 Unknown database 'none'`
- **原因**：数据库名读成 `None`，f-string 拼进 URL 变 `"None"`，或某处默认值写死 `"none"`
- **解决**：
  1. 修 key 名对齐
  2. MySQL 里 `CREATE DATABASE student_manage_db`
- **避免**：同上，`_require` 校验 + 建库 SQL 存档

#### A4. 日志文件变成 `student.log.log`

- **现象**：`.env` 写 `LOG_NAME=student.log`，日志文件名变 `student.log.log`
- **原因**：`logger.py` 里会拼 `.log`，`LOG_NAME` 又带了一次
- **解决**：`.env` 里改成 `LOG_NAME=student`
- **避免**：`LOG_NAME` 只放名字，扩展名由代码统一加

#### A5. `Level not an integer or a valid string: None`

- **现象**：`logger.setLevel(None)` 报错
- **原因**：`.env` 里没配 `LOG_LEVEL`
- **解决**：`LOG_LEVEL = os.getenv("LOG_LEVEL", "INFO")`
- **避免**：所有日志相关变量给默认值

### B. 数据库连接 / 会话类

#### B1. `thread_local` 缓存已关闭的 session

- **现象**：第一次调用没问题，第二次拿到坏 session，报各种 `Session is closed`
- **原因**：`get_session` 用 thread_local 缓存，但 dao 里 `session.close()` 后 thread_local 里还留着引用
- **解决**：`get_session` 每次返回全新 session，不做缓存

```python
@classmethod
def get_session(cls):
    cls._init_engine()
    return cls.__SessionLocal()
```

- **避免**：除非有性能瓶颈，否则不要缓存 session；连接池已经把连接复用的收益吃掉了

#### B2. 忘记 `session.close()` 导致连接泄漏

- **现象**：程序跑久了连接耗尽，MySQL 报 `Too many connections`
- **原因**：某个 dao 方法异常时没走到 `close`
- **解决**：所有 dao 方法用 `try / except / finally` 包住，`close` 放 finally
- **避免**：dao 模板化，复制粘贴，别手写

#### B3. 忘记 `session.commit()` 数据没落库

- **现象**：日志显示"插入成功"，但数据库里查不到
- **原因**：`execute` 后没 `commit`，程序退出时事务回滚
- **解决**：DML 方法（insert/update/delete）执行后必须 `commit`
- **避免**：
  - 用 `session_dml` 上下文管理器（with 自动 commit）
  - 或 DAO 模板固定带 commit

#### B4. 重复 commit

- **现象**：`with DBUtil.session_dml()` 里调用 dao，dao 内部也 commit，可能出现"已关闭 session 被 commit"
- **原因**：事务边界不清晰——两处都在管
- **解决**：二选一
  - **方案 A（推荐）**：DAO 管事务，service 直接调
  - **方案 B**：Service 管事务（`@transactional`），DAO 只 execute
- **避免**：项目开始时定好事务边界，别混用

#### B5. 事务不原子

- **现象**：一个业务涉及多次写操作，前一次成功、后一次失败，数据不一致
- **原因**：每次 dao 调用各自 commit，没有统一事务
- **解决**：用 `@transactional` 装饰 service 方法，所有 dao 共用一个 session
- **避免**：组合操作时一定要想"事务覆盖范围"

### C. SQLAlchemy / SQL 类

#### C1. `'CursorResult' object has no attribute 'inserted_id'`

- **现象**：insert 后取主键报错
- **原因**：`inserted_id` 是 PyMongo 的 API，SQLAlchemy 里没有
- **解决**：
  - MySQL 用 `result.lastrowid`
  - 或通用写法 `result.inserted_primary_key[0]`
- **避免**：别把 MongoDB 的习惯带过来

#### C2. 返回 `lastrowid` 而不是 `rowcount`

- **现象**：service 判断 `row <= 0` 失败
- **原因**：insert 返回的是主键 ID（可能大于 0），不是受影响行数
- **解决**：insert 方法最后 `return result.rowcount`
- **避免**：所有 dao 方法返回值语义统一——insert/update/delete 都返回 rowcount

#### C3. SQL 里 `: param` 中间有空格

- **现象**：`UPDATE ... SET update_time=: update_time` 参数绑定失败
- **原因**：SQLAlchemy 参数名紧跟冒号，不能有空格
- **解决**：`:update_time`
- **避免**：写 SQL 时贴着写，或让 IDE 高亮帮你看

#### C4. 动态 SQL 参数缩进错位

```python
if score.stu_id is not None:
    sql += ", `stu_id` = :stu_id"
params["stu_id"] = score.stu_id       # ← 缩进在外面！无条件执行
```

- **现象**：SQL 里没这个字段，params 里却有 → SQLAlchemy 报参数不匹配
- **解决**：参数赋值紧跟 SQL 拼接，缩进一致
- **避免**：动态拼 SQL 时，SQL 和 params 成对写在一起

#### C5. `_COLUMNS` 未定义

- **现象**：`SELECT {XxxDao._COLUMNS}` 报 `NameError`
- **原因**：类属性没定义
- **解决**：类顶部加 `_COLUMNS = "..."`
- **避免**：dao 模板固定包含 `_COLUMNS`，所有查询用它

#### C6. LIKE 模糊查询误当唯一性校验

- **现象**：新增课程 "Python" 时，因为库里已有 "Python程序设计"，报"课程已存在"
- **原因**：`select_by_name` 用 LIKE，用于搜索，不该用于唯一性校验
- **解决**：新增 `select_by_name_exact` 用 `=` 精确匹配
- **避免**：搜索方法和唯一性校验方法分开，命名上区分

### D. 日志类

#### D1. `log.error("xxx", e)` 导致 Logging error

```
TypeError: not all arguments converted during string formatting
```

- **原因**：logging 第二个参数是格式化参数，消息里没 `%s` 却传了参数
- **解决**：三种写法任选
  ```python
  log.exception("插入失败")             # 推荐，自动带堆栈
  log.error("插入失败: %s", e)
  log.error(f"插入失败: {e}")
  ```
- **避免**：别用 `log.error("msg", e)` 这种"想当然"的写法

#### D2. 日志文件重复 `.log`

- 见 A4

#### D3. `logger.setLevel(None)`

- 见 A5

### E. 异常处理类

#### E1. `raise DatabaseError(e)` 丢失原始堆栈

- **现象**：只能看到 `DatabaseError`，原始 `OperationalError` 被吞
- **原因**：没加 `from e`
- **解决**：`raise DatabaseError(e) from e`
- **避免**：所有异常包装都带 `from e`

#### E2. 异常类型不统一

- **现象**：`clazz_service` 用 `ValidationError`，`student_service` 用 `ValueError`，`main` 里得捕获两种
- **原因**：没定异常体系
- **解决**：
  - 定义 `core.exceptions`，分 `AppException` → `BusinessException` / `SystemException`
  - Service 抛业务异常，DAO 抛 `DatabaseError`
  - `main.py` 里 `except AppException` 统一处理
- **避免**：项目开始时设计好异常层级

#### E3. `AlreadyExistsError` / `ValidationError` 继承错位

- **现象**：`NotFoundError` 是 `BusinessException` 子类，但 `AlreadyExistsError` 挂在 `AppException` 下
- **原因**：异常层次定义时不严谨
- **解决**：所有业务异常都继承 `BusinessException`
- **避免**：定义异常时先画继承树

### F. 业务校验类

#### F1. 唯一性校验只在 add 里做，update 忘了

- **现象**：更新课程名时改成和别的课同名，能成功
- **原因**：`update` 缺唯一性校验
- **解决**：`update` 里也检查字段唯一性，注意排除自己：

```python
exist = Dao.select_by_xxx(new_value)
if exist is not None and exist.id != current.id:
    raise AlreadyExistsError(...)
```

- **避免**：所有涉及唯一字段的更新都要加

#### F2. 关联存在性校验只在 add 里做

- **现象**：更新成绩把 `stu_id` 改成不存在的值，能成功
- **原因**：`update` 没校验外键存在
- **解决**：`update` 里对所有可能改动的外键字段做存在性校验
- **避免**：add/update 用同一套校验函数

#### F3. class_id = 0 被当成有效值

- **现象**：用户输入 0，`0 is not None` 为 True，被写入库
- **原因**：CLI 层把空输入转成 0
- **解决**：
  - CLI 层空 → `None`
  - Service 层校验 `class_id <= 0` 报错
- **避免**：约定 0 不是有效 ID；CLI 空输入用 None

#### F4. 字段类型和范围没校验

- **现象**：age 输成 999，手机号 "abc" 都能写进去
- **原因**：Service 缺校验
- **解决**：抽出 `_validate_age` / `_validate_phone` 等，add 和 update 都调
- **避免**：每个字段想清楚"合理取值范围"，写进校验函数

### G. Python / 项目结构类

#### G1. 文件被误覆盖

- **现象**：`model/course.py` 内容变成 `CourseDao`
- **原因**：多文件操作时复制粘贴串了
- **解决**：检查文件内容是否与文件名相符
- **避免**：给文件加明显的 docstring 头，一眼能看出是什么

#### G2. 文件名用 `.` 而不是 `_`

- **现象**：`service/student.service.py`
- **原因**：命名不规范
- **解决**：重命名为 `student_service.py`
- **避免**：Python 模块名只用小写 + 下划线

#### G3. 循环导入

- **现象**：`from model.course import Course` 在 `model/course.py` 里
- **原因**：文件名和 import 对不上
- **解决**：确认 import 路径正确
- **避免**：层级清晰，model 不 import dao

#### G4. 文件头 docstring 重复

- **现象**：顶部两个 `"""..."""` 连着
- **解决**：删掉一个
- **避免**：复制模板时注意去掉重复

### H. CLI 交互类

#### H1. 空输入变成 0

- 见 F3

#### H2. `int()` 抛 ValueError 飘到最外层

- **现象**：用户输入字母，程序报 `[系统错误] invalid literal for int()`
- **解决**：

```python
def input_int(prompt, allow_empty=False):
    raw = input(prompt).strip()
    if raw == "":
        if allow_empty:
            return None
        raise ValueError("需要整数")
    return int(raw)
```

main 里 `except ValueError` 给友好提示

- **避免**：输入转换统一走工具函数

#### H3. 更新时无法清空字段

- **现象**：update 时输入空回车 → `None` → DAO 跳过更新 → 字段不会变 NULL
- **原因**：用 `None` 表示"不修改"，就没办法表示"清空"
- **解决**（可选）：约定特殊符号，如输入 `-` 表示清空
- **避免**：设计 CLI 时想清楚"不修改"和"清空"怎么区分

---

## 六、关键设计原则

### 6.1 单向依赖

```
main → service → dao → model
              ↘ core (日志、异常)
              ↘ utils (db_util, env_util)
```

**下层不知道上层存在。** model 不 import dao，dao 不 import service。

### 6.2 事务边界清晰

- **推荐**：DAO 管事务（自己 commit/rollback/close）
- **进阶**：Service 用 `@transactional` 管，DAO 只 execute
- **禁忌**：两处都管 → 重复 commit

### 6.3 异常分层

```
AppException（基类）
├── BusinessException（业务，4xx）
│   ├── NotFoundError
│   ├── AlreadyExistsError
│   ├── ValidationError
│   └── OperationFailedError
├── SystemException（系统，5xx）
│   ├── DatabaseError
│   └── ConfigError
├── UnauthorizedError（401）
└── ForbiddenError（403）
```

**Service 抛 BusinessException，DAO 抛 DatabaseError，main 统一 catch AppException。**

### 6.4 参数绑定，杜绝 SQL 注入

```python
# ✅ 正确
session.execute(text("SELECT * FROM t WHERE id = :id"), {"id": user_input})

# ❌ 错误（会 SQL 注入）
session.execute(text(f"SELECT * FROM t WHERE id = {user_input}"))
```

**永远用 `:param` + 字典传参，不要 f-string 拼用户输入。**

### 6.5 校验一致

- **add 和 update 用同一套校验函数**（抽 `_validate_xxx`）
- **唯一性校验排除自己**（`exist.id != current.id`）
- **空字符串 vs None** 统一处理

### 6.6 日志规范

- **DAO 层**：`log.exception("xxx失败")`，自动带堆栈
- **Service 层**：`log.info("xxx成功 id=%s", id)`，关键动作记录
- **CLI 层**：不打日志，只 `print` 给用户看
- **格式**：`%s` 占位符，不要 f-string 传参给 log

---

## 七、快速自查清单

### 环境配置

- [ ] `.env` 在项目根目录
- [ ] 所有 key 与代码里 `getenv` 一致
- [ ] 关键变量有默认值或启动校验
- [ ] MySQL 里数据库已创建

### DAO 层

- [ ] 每个 DML 方法都有 `commit`
- [ ] 每个方法都有 `try / except / finally`
- [ ] `finally` 里 `close`
- [ ] insert 返回 `rowcount`，主键用 `lastrowid` 回填
- [ ] 所有查询有 `_COLUMNS` 常量

### Service 层

- [ ] `add` / `update` 都有必填校验
- [ ] `update` 有存在性校验 + 唯一性校验（排除自己）
- [ ] 关联字段（外键）有存在性校验
- [ ] 时间字段自动填
- [ ] 异常用 `core.exceptions` 里的类型

### 异常

- [ ] `raise XxxError(e) from e` 保留原因链
- [ ] `main.py` 里 `except AppException` 统一处理
- [ ] 业务异常不打印堆栈，系统异常打印

### 日志

- [ ] `log.error` 用 `%s` 或 `log.exception`
- [ ] `LOG_NAME` 不带 `.log`
- [ ] `LOG_LEVEL` 有默认值

### CLI

- [ ] 输入转换工具函数统一处理空值和类型错误
- [ ] 关键操作有确认（如删除）
- [ ] 空输入转 None 而不是 0

---

## 八、常用代码片段

### 8.1 环境变量工具

```python
from dotenv import load_dotenv
import os

load_dotenv()

def _require(key):
    val = os.getenv(key)
    if not val:
        raise RuntimeError(f"环境变量 {key} 未设置")
    return val

class EnvUtil:
    DB_HOST = _require("DB_HOST")
    DB_PORT = int(_require("DB_PORT"))
    DB_USER = _require("DB_USER")
    DB_PASSWORD = _require("DB_PASSWORD")
    DB_DATABASE = _require("DB_DATABASE")
    DB_CHARSET = os.getenv("DB_CHARSET", "utf8mb4")

    LOG_LEVEL = os.getenv("LOG_LEVEL", "INFO")
    LOG_DIR = os.getenv("LOG_DIR", "logs")
    LOG_NAME = os.getenv("LOG_NAME", "student")
```

### 8.2 数据库工具

```python
import threading
from sqlalchemy import create_engine
from sqlalchemy.orm import sessionmaker, Session
from utils.env_util import EnvUtil


class DBUtil:
    __engine = None
    __SessionLocal = None
    __lock = threading.Lock()

    @classmethod
    def _init_engine(cls):
        if cls.__engine is None:
            with cls.__lock:
                if cls.__engine is None:
                    url = (
                        f"mysql+pymysql://{EnvUtil.DB_USER}:{EnvUtil.DB_PASSWORD}"
                        f"@{EnvUtil.DB_HOST}:{EnvUtil.DB_PORT}/{EnvUtil.DB_DATABASE}"
                        f"?charset={EnvUtil.DB_CHARSET}"
                    )
                    cls.__engine = create_engine(
                        url,
                        pool_size=5,
                        max_overflow=10,
                        pool_timeout=30,
                        pool_recycle=3600,
                        pool_pre_ping=True,
                        echo=False,
                    )
                    cls.__SessionLocal = sessionmaker(
                        bind=cls.__engine,
                        autoflush=False,
                        autocommit=False,
                    )

    @classmethod
    def get_session(cls) -> Session:
        cls._init_engine()
        return cls.__SessionLocal()
```

### 8.3 事务装饰器

```python
from functools import wraps


def transactional(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        session = DBUtil.get_session()
        try:
            result = func(*args, **kwargs)
            session.commit()
            return result
        except Exception:
            session.rollback()
            raise
        finally:
            session.close()
    return wrapper
```

### 8.4 DAO 模板

```python
class XxxDao:
    _COLUMNS = "`id`, `name`, ..."

    @staticmethod
    def insert(xxx):
        session = None
        try:
            sql = "INSERT INTO ..."
            params = {...}
            session = DBUtil.get_session()
            result = session.execute(text(sql), params)
            session.commit()
            xxx.id = result.lastrowid
            return result.rowcount
        except Exception as e:
            if session:
                session.rollback()
            log.exception("插入xxx失败")
            raise DatabaseError(e) from e
        finally:
            if session:
                session.close()
```

### 8.5 Service 模板

```python
@staticmethod
def add(xxx):
    # 1. 必填校验
    if not xxx.name:
        raise ValidationError("xxx名称不能为空", field="name")
    # 2. 唯一性校验
    if XxxDao.select_by_name(xxx.name):
        raise AlreadyExistsError("xxx", field="name", value=xxx.name)
    # 3. 关联校验
    if xxx.parent_id and not ParentDao.select_by_id(xxx.parent_id):
        raise NotFoundError("父级", xxx.parent_id)
    # 4. 填时间
    now = datetime.now()
    xxx.create_time = now
    xxx.update_time = now
    # 5. 调 DAO
    row = XxxDao.insert(xxx)
    if row <= 0:
        raise BusinessException("新增xxx失败")
    log.info("新增xxx成功 id=%s", xxx.id)
    return row
```

### 8.6 main.py 异常处理模板

```python
try:
    # 业务调用
    XxxService.add(xxx)
except AppException as e:
    print(f"[业务错误] {e.message}")
except ValueError as e:
    print(f"[输入错误] {e}")
except Exception as e:
    print(f"[系统错误] {e}")
```

### 8.7 建库 SQL 模板

```sql
DROP DATABASE IF EXISTS student_manage_db;
CREATE DATABASE student_manage_db
    CHARACTER SET utf8mb4
    COLLATE utf8mb4_0900_ai_ci;

USE student_manage_db;

CREATE TABLE `t_class` (
    `id`           BIGINT       NOT NULL AUTO_INCREMENT COMMENT '主键ID',
    `name`         VARCHAR(50)  NOT NULL COMMENT '班级名称',
    `teacher_name` VARCHAR(30)  NOT NULL COMMENT '班主任',
    `create_time`  DATETIME     NULL DEFAULT CURRENT_TIMESTAMP,
    `update_time`  DATETIME     NULL DEFAULT CURRENT_TIMESTAMP,
    PRIMARY KEY (`id`)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4;

-- 加唯一索引兜底
ALTER TABLE `t_class` ADD UNIQUE INDEX `uk_class_name`(`name`);
```

---

## 九、项目可扩展方向

### 9.1 事务进阶

引入 `@transactional` 装饰器，让 Service 管事务，DAO 只 execute，支持跨多个 DAO 的原子操作（如"学生转班"：更新学生 + 更新班级人数）。

### 9.2 密码加密

目前 `t_user.password` 是明文。生产环境应用 `bcrypt` / `passlib`：

```python
import bcrypt

hashed = bcrypt.hashpw(password.encode(), bcrypt.gensalt())
bcrypt.checkpw(input_pw.encode(), hashed)
```

### 9.3 分页查询

`select_all` 在数据量大时慢。加 `LIMIT / OFFSET`：

```python
def select_page(self, page=1, size=20):
    offset = (page - 1) * size
    sql = "SELECT ... LIMIT :size OFFSET :offset"
```

### 9.4 唯一索引兜底

应用层做了唯一性校验，但并发下仍可能重复。数据库加 UNIQUE 索引：

```sql
ALTER TABLE `t_course` ADD UNIQUE INDEX `uk_course_name`(`name`);
ALTER TABLE `t_class` ADD UNIQUE INDEX `uk_class_name`(`name`);
```

报 `Duplicate entry` 时 `DatabaseError` 捕获。

### 9.5 数据软删除

不物理删除，加 `is_deleted TINYINT` 字段，查询带 `WHERE is_deleted = 0`。保留历史数据。

### 9.6 Web 化

- 把 `main.py` 换成 Flask / FastAPI
- Service 层不用改
- 异常 `to_dict()` 直接返回 JSON
- 加 JWT 认证

### 9.7 缓存

热点数据（如课程列表）用 Redis 缓存，减少数据库压力。查询先看缓存，miss 再查库并回写。

### 9.8 单元测试

用 `pytest` + `unittest.mock`：

```python
def test_student_add_duplicate_stu_num():
    """学号重复应该报 AlreadyExistsError"""
    with pytest.raises(AlreadyExistsError):
        StudentService.add(Student(stu_num="S001", name="张三"))
```

---

## 十、一句话总结

> **下层稳定、上层灵活；事务边界清晰、异常分层明确；参数绑定防注入、校验一致防漏网；日志规范、DAO 模板化，剩下的就是重复劳动。**

按这个思路走，三层架构的项目基本不会出大问题。新功能只需在对应的层加方法，不用动其他层。

---

## 附：项目完成度记录

| 模块 | Model | DAO | Service | main 菜单 | 状态 |
|---|---|---|---|---|---|
| 班级 Clazz | ✓ | ✓ 6 方法 | ✓ 6 方法 | ✓ | ✅ 完成 |
| 学生 Student | ✓ | ✓ 7 方法 | ✓ 7 方法 | ✓ | ✅ 完成 |
| 课程 Course | ✓ | ✓ 6 方法 | ✓ 5 方法 | ✓ | ✅ 完成 |
| 成绩 Score | ✓ | ✓ 7 方法 | ✓ 7 方法 | ✓ | ✅ 完成 |
| 用户 User | ✓ | ✓ 1 方法 | ✓ 1 方法 | ✓ | ✅ 完成 |
| 日志 logger | — | — | — | — | ✅ 完成 |
| 异常 exceptions | — | — | — | — | ✅ 完成 |
| 数据库工具 DBUtil | — | — | — | — | ✅ 完成 |
| 环境变量 EnvUtil | — | — | — | — | ✅ 完成 |

**项目全部功能已跑通：登录 → 班级管理 → 学生管理 → 课程管理 → 成绩管理，所有增删改查和边界校验都正常。**

